# Phase 0 — EDA & Preprocessing
**DiagGen+ | Advanced AI Group Project 2026**

## Objectives
1. Load and inspect the dataset.
2. Analyse class distribution — identify rare disease classes.
3. Run the NLP preprocessing pipeline and verify output quality.
4. Produce the 80/10/10 train/val/test split and save to `data/processed/`.

> **Gate 0 complete** when: clean dataset exists, class distribution chart saved, preprocessing verified.

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils.config_loader import cfg, get
from src.utils.logger import get_logger
from src.preprocessing.cleaner import preprocess, preprocess_batch
from src.training.trainer import load_dataset, split_data, build_label_maps

logger = get_logger('phase0_eda')
FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

## 1. Load Dataset

In [ ]:
# Update path to match your downloaded file (see data/README.md)
DATA_PATH = Path('../data/raw/dataset.csv')

df = load_dataset(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 2. Class Distribution — Identify Rare Classes

In [ ]:
counts = df['disease'].value_counts()
rare_threshold = get('data.rare_class_threshold', 100)

print(f'Total classes:  {len(counts)}')
print(f'Rare classes:   {(counts < rare_threshold).sum()}  (< {rare_threshold} samples)')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colours = ['#E74C3C' if c < rare_threshold else '#2E75B6' for c in counts.values]
ax.bar(counts.index, counts.values, color=colours)
ax.axhline(y=rare_threshold, color='orange', linestyle='--',
           label=f'Rare threshold ({rare_threshold})')
ax.set_xlabel('Disease')
ax.set_ylabel('Sample count')
ax.set_title('Class Distribution — Red = rare classes targeted for VAE augmentation')
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.show()

## 3. Preprocessing Verification

In [ ]:
# Spot-check on 10 samples
samples = df['symptoms'].sample(10, random_state=42).tolist()
for orig in samples:
    proc = preprocess(orig)
    print(f'ORIGINAL:  {orig[:100]}')
    print(f'PROCESSED: {proc[:100]}')
    print()

In [ ]:
# Batch preprocess full dataset
print('Batch preprocessing...')
df['symptoms_clean'] = preprocess_batch(df['symptoms'].tolist())
print('Done.')
df[['symptoms','symptoms_clean','disease']].head()

## 4. Train / Val / Test Split

In [ ]:
train_df, val_df, test_df = split_data(df)
label2id, id2label, le   = build_label_maps(df)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Classes: {len(label2id)}')

processed_dir = Path('../data/processed')
train_df.to_csv(processed_dir / 'train.csv', index=False)
val_df.to_csv(processed_dir   / 'val.csv',   index=False)
test_df.to_csv(processed_dir  / 'test.csv',  index=False)
print('Splits saved.')

## 5. Phase 0 Summary

Fill in after running:

| Metric | Value |
|--------|-------|
| Total samples | — |
| Disease classes | — |
| Rare classes | — |
| Train / Val / Test | — / — / — |

> Copy this table into the report Methodology section.